In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_50_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 51 ###

# Ensure the question string is in scope (avoid NameError)
question_of_interest_cell63 = 'What is your current yearly compensation (approximate $USD)?'

# Define the desired salary buckets in order
responses_in_order_cell63 = [
    '$0-999', '1,000-1,999', '2,000-2,999', '3,000-3,999',
    '5,000-7,499', '7,500-9,999', '10,000-14,999', '15,000-19,999',
    '20,000-24,999', '25,000-29,999', '30,000-39,999', '40,000-49,999',
    '50,000-59,999', '60,000-69,999', '70,000-79,999', '80,000-89,999',
    '90,000-99,999', '100,000-124,999', '125,000-149,999', '150,000-199,999',
    '200,000-249,999', '250,000-299,999', '300,000-499,999', '$500,000-999,999',
    '>$1,000,000'
]

# Boolean mask for U.S. respondents
us_mask = (
    responses_df_2022_cell10['In which country do you currently reside?']
    == 'United States of America'
)

# Extract the salary column for U.S. respondents and cast to ordered categorical
salaries_us = (
    responses_df_2022_cell10.loc[us_mask, question_of_interest_cell63]
    .astype('category')
    .cat.set_categories(responses_in_order_cell63, ordered=True)
)

# Compute percentage breakdown directly on the GPU
percentages_cell63 = (
    salaries_us.value_counts(dropna=False, normalize=True, sort=False)
    * 100
).round(1)

# Reindex to enforce the specified order
percentages_cell63 = percentages_cell63.reindex(responses_in_order_cell63)

# Display the SERIES information
percentages_cell63.info()